# 🏭 Project 22 · AGV Route Intelligence
## Tabular Q-Learning for Autonomous Navigation in a Factory Floor

**Reinforcement Learning Family | Operational Excellence Portfolio | LozanoLsa**

---

For twenty-one projects, the models in this portfolio learned to **observe**.

They classified failures, predicted outcomes, and detected anomalies —
always given historical data, always asked to find patterns that already existed.
Their intelligence was retrospective: trained on what had already happened.

**This notebook changes the paradigm.**

The agent here receives no dataset of labeled examples.
It receives a **world**: a 10×10 factory floor with obstacles, congestion zones, and a destination dock.
It receives a **reward signal**: +100 for reaching the goal, −1 per step wasted, −50 for hitting a wall,
−10 for crossing a congested aisle.
And a single mandate: **learn to navigate — on your own.**

Q-Learning is how a machine learns from consequence rather than example.
Every episode, the agent starts somewhere, makes decisions, collects rewards, and updates its memory.
After 5,000 episodes, it has transformed from blind randomness into purposeful navigation.

This is not a small shift in technique. It is a shift in the nature of machine intelligence:
from pattern recognition to **autonomous decision-making**.

---

## The Industrial Problem

An AGV (Automated Guided Vehicle) must navigate a factory floor to deliver parts
to a docking station. The floor has:

| Element | Description | RL Meaning |
|---------|-------------|-----------|
| **10×10 grid** | Factory floor cells | State space (100 states) |
| **4 actions** | Up / Down / Left / Right | Action space |
| **10 obstacles** | Fixed walls and equipment | Reward = −50, position unchanged |
| **3 congestion zones** | High-traffic aisles | Extra penalty of −10 per crossing |
| **1 goal** | Dock at (9,9) | Reward = +100, episode ends |
| **Step cost** | Time is money | −1 per move |

The agent doesn't know any of this at the start. It learns it all through consequences.

---
**LozanoLsa | Operational Excellence · MBB · Machine Learning | GitHub: LozanoLsa**


## 2 · Setup — The Tools of Navigation

In [ ]:
# ── 2 · Setup — The Tools of Navigation ─────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from typing import List, Tuple

# ── Dark palette ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor":   "#161b22",
    "axes.edgecolor":   "#30363d",
    "axes.labelcolor":  "#c9d1d9",
    "xtick.color":      "#8b949e",
    "ytick.color":      "#8b949e",
    "text.color":       "#c9d1d9",
    "grid.color":       "#21262d",
    "grid.linestyle":   "--",
    "grid.alpha":       0.4,
    "legend.facecolor": "#161b22",
    "legend.edgecolor": "#30363d",
})

C_BLUE = "#4C9BE8"
C_RED  = "#E8574C"
C_OK   = "#3FB950"
C_WARN = "#F5A623"
C_BG   = "#0d1117"
C_CARD = "#161b22"

print("Q-Learning AGV pipeline initialized.")
print("  Paradigm shift: from pattern recognition to autonomous decision-making.")

## 3 · The World — A Factory Floor in 10 × 10

Before the agent can learn, we must define the world it will inhabit.

The environment is a **Markov Decision Process (MDP)**:
- **States** $s \in \{0, 1, \ldots, 99\}$: each cell of the 10×10 grid
- **Actions** $a \in \{0, 1, 2, 3\}$: up, down, left, right
- **Transition** $T(s, a) \to s'$: deterministic (no stochasticity)
- **Reward** $R(s, a, s')$: defined by the reward function below
- **Terminal state**: goal cell (9, 9)

### Reward Function

| Situation | Reward |
|-----------|--------|
| Reach goal (9,9) | **+100** |
| Hit wall or obstacle | **−50** (stay in place) |
| Cross congestion zone | **−1 − 10 = −11** (step + penalty) |
| Normal move | **−1** (step cost) |

The negative step cost is essential: it pushes the agent toward the **shortest** path,
not just any path that eventually reaches the goal.


In [ ]:
# ── Environment class — AGVGridWorld ─────────────────────────────────────────
class AGVGridWorld:
    """
    10x10 grid-world environment for an AGV routing problem.
    States  : cells (row, col) encoded as integers 0..99
    Actions : 0=up, 1=down, 2=left, 3=right
    """
    def __init__(self, rows: int = 10, cols: int = 10):
        self.rows, self.cols = rows, cols
        self.n_states  = rows * cols
        self.n_actions = 4
        self.action_names = ["up", "down", "left", "right"]

        # Fixed obstacles (walls, equipment)
        self.obstacles = {
            (2,4),(3,4),(4,4),(5,4),(6,4),(7,4),   # vertical wall
            (1,7),(3,2),(5,7),(7,2)                  # scattered blocks
        }
        self.goal = (9, 9)                            # delivery dock
        self.congestion_zones = {(4,5),(4,6),(4,7)}  # high-traffic aisles

    def state_to_rc(self, s): return s // self.cols, s % self.cols
    def rc_to_state(self, r, c): return r * self.cols + c

    def is_valid(self, r, c):
        return 0 <= r < self.rows and 0 <= c < self.cols and (r,c) not in self.obstacles

    def sample_start(self):
        valid = [s for s in range(self.n_states)
                 if self.state_to_rc(s) != self.goal
                 and self.state_to_rc(s) not in self.obstacles]
        return int(np.random.choice(valid))

    def step(self, state, action):
        r, c = self.state_to_rc(state)
        dr = [-1, 1, 0, 0]; dc = [0, 0, -1, 1]
        nr, nc = r + dr[action], c + dc[action]

        if not self.is_valid(nr, nc):
            return state, -50.0, False
        if (nr, nc) == self.goal:
            return self.rc_to_state(nr, nc), 100.0, True

        reward = -1.0
        if (nr, nc) in self.congestion_zones:
            reward -= 10.0
        return self.rc_to_state(nr, nc), reward, False

env = AGVGridWorld()
print(f"Environment ready: {env.rows}x{env.cols} = {env.n_states} states, {env.n_actions} actions")
print(f"Goal: {env.goal}  |  Obstacles: {len(env.obstacles)}  |  Congestion zones: {len(env.congestion_zones)}")

In [ ]:
# ── Visualize the factory floor ──────────────────────────────────────────────
def plot_grid(env, path=None, title="Factory Floor Layout", ax=None):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(6.5, 6.5))
        fig.patch.set_facecolor(C_BG)

    ax.set_facecolor(C_CARD)
    ax.set_xlim(0, env.cols); ax.set_ylim(0, env.rows)

    # Grid lines
    for i in range(env.rows + 1):
        ax.axhline(i, color="#30363d", lw=0.5)
    for j in range(env.cols + 1):
        ax.axvline(j, color="#30363d", lw=0.5)

    # Obstacles
    for (r,c) in env.obstacles:
        ax.fill_between([c, c+1], r, r+1, color="#E8574C", alpha=0.75, zorder=2)

    # Congestion zones
    for (r,c) in env.congestion_zones:
        ax.fill_between([c, c+1], r, r+1, color=C_WARN, alpha=0.40, zorder=2)

    # Goal
    gr, gc = env.goal
    ax.fill_between([gc, gc+1], gr, gr+1, color=C_OK, alpha=0.7, zorder=2)
    ax.text(gc+0.5, gr+0.5, "G", ha="center", va="center",
            fontsize=13, fontweight="bold", color="white", zorder=4)

    # Path
    if path:
        xs = [c+0.5 for (r,c) in path]
        ys = [r+0.5 for (r,c) in path]
        ax.plot(xs, ys, color=C_BLUE, lw=2.5, zorder=5, marker="o",
                markersize=5, markerfacecolor=C_BLUE)
        sr, sc = path[0]
        ax.text(sc+0.5, sr+0.5, "S", ha="center", va="center",
                fontsize=11, fontweight="bold", color="white", zorder=6)

    ax.invert_yaxis()
    ax.set_xticks([j+0.5 for j in range(env.cols)])
    ax.set_xticklabels(range(env.cols), fontsize=8)
    ax.set_yticks([i+0.5 for i in range(env.rows)])
    ax.set_yticklabels(range(env.rows), fontsize=8)
    ax.set_title(title, color="white", fontsize=11, pad=8)

    legend = [
        mpatches.Patch(color=C_RED,  alpha=0.75, label="Obstacle"),
        mpatches.Patch(color=C_WARN, alpha=0.40, label="Congestion zone"),
        mpatches.Patch(color=C_OK,   alpha=0.70, label="Goal (9,9)"),
    ]
    ax.legend(handles=legend, fontsize=8, loc="upper right")

    if standalone:
        plt.tight_layout()
        plt.savefig("fig_01_grid.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
        plt.show()

plot_grid(env, title="Factory Floor — 10x10 AGV Navigation Environment")
print("Red = obstacles | Orange = congestion zones | Green = goal dock")

## 4 · The Algorithm — Learning from Consequence

### Q-Learning: The Core Idea

The agent maintains a **Q-table**: a matrix $Q(s, a)$ of size $100 \times 4$.
Each entry estimates the **expected total future reward** of taking action $a$ in state $s$.

At the start, $Q(s,a) = 0$ for all states and actions — the agent knows nothing.

After each action, the agent receives a reward and updates its estimate:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \cdot \underbrace{\left[ r + \gamma \cdot \max_{a'} Q(s',a') - Q(s,a) \right]}_{\text{TD error } \delta}$$

Where:
- $\alpha = 0.1$ — learning rate (how fast to update estimates)
- $\gamma = 0.95$ — discount factor (future rewards are worth 95% of immediate ones)
- $r$ — reward received
- $s'$ — next state
- $\delta$ — TD error: the surprise signal

### Exploration vs. Exploitation: $\varepsilon$-Greedy

Early episodes: the agent explores randomly ($\varepsilon = 1.0$).
Over time, $\varepsilon$ decays ($\varepsilon \times 0.995$ per episode) until it reaches $\varepsilon_{min} = 0.05$.
At convergence, the agent exploits its learned Q-values 95% of the time.

**The journey**: 5,000 episodes. Episode 1: blind wandering. Episode 5,000: confident navigation.


In [ ]:
# ── Q-Learning algorithm ─────────────────────────────────────────────────────
def q_learning(env, episodes=5000, alpha=0.1, gamma=0.95,
               epsilon=1.0, epsilon_min=0.05, epsilon_decay=0.995, max_steps=200):
    """
    Train a Q-table using the Q-Learning algorithm.

    Parameters
    ----------
    env          : AGVGridWorld — the environment
    episodes     : int — total training episodes
    alpha        : float — learning rate
    gamma        : float — discount factor
    epsilon      : float — initial exploration rate
    epsilon_min  : float — minimum exploration rate
    epsilon_decay: float — multiplicative decay per episode
    max_steps    : int — maximum steps per episode

    Returns
    -------
    Q      : np.ndarray (n_states x n_actions) — learned Q-table
    rewards : list[float] — total reward per episode
    """
    Q = np.zeros((env.n_states, env.n_actions))
    rewards = []

    for ep in range(episodes):
        state = env.sample_start()
        ep_reward = 0.0

        for _ in range(max_steps):
            # epsilon-greedy action selection
            if np.random.rand() < epsilon:
                action = np.random.randint(env.n_actions)   # explore
            else:
                action = int(np.argmax(Q[state, :]))         # exploit

            next_state, reward, done = env.step(state, action)

            # Q-update (Bellman equation)
            td_target = reward + gamma * np.max(Q[next_state, :])
            td_error   = td_target - Q[state, action]
            Q[state, action] += alpha * td_error

            state = next_state
            ep_reward += reward
            if done:
                break

        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards.append(ep_reward)

    return Q, rewards


def build_qtable_df(Q, env):
    """Convert Q-table to a readable DataFrame."""
    records = []
    for s in range(env.n_states):
        r, c = env.state_to_rc(s)
        best_idx = int(np.argmax(Q[s, :]))
        records.append({
            "state": s, "row": r, "col": c,
            "Q_up": Q[s,0], "Q_down": Q[s,1],
            "Q_left": Q[s,2], "Q_right": Q[s,3],
            "best_action": env.action_names[best_idx],
            "best_Q": float(Q[s, best_idx])
        })
    return pd.DataFrame(records)

print("Q-Learning algorithm defined.")
print("  Update rule : Q(s,a) += alpha * [r + gamma * max Q(s') - Q(s,a)]")
print("  Exploration : epsilon-greedy (eps: 1.0 -> 0.05 over 5,000 episodes)")

## 5 · Training — 5,000 Episodes of Trial and Error

The agent starts each episode at a random valid cell and navigates until it reaches the goal
or exhausts its 200-step budget.

**What to expect:**
- **Early episodes**: random paths, massive penalties for wall collisions → large negative rewards
- **Episode ~236**: the rolling average first crosses zero — the agent starts reaching the goal reliably
- **Episode 5000**: epsilon=0.05, mean reward ≈ 85 — optimal routing achieved

The transformation from −820 (first 100 episodes) to +85 (last 100 episodes)
is the numerical signature of learning.


In [ ]:
# ── Train the agent ──────────────────────────────────────────────────────────
np.random.seed(42)   # for reproducible training

Q, episode_rewards = q_learning(
    env,
    episodes     = 5000,
    alpha        = 0.1,
    gamma        = 0.95,
    epsilon      = 1.0,
    epsilon_min  = 0.05,
    epsilon_decay= 0.995,
    max_steps    = 200
)

print(f"Training complete. Q-table shape: {Q.shape}")
print()
print(f"{'Phase':<25} {'Episodes':>10} {'Mean Reward':>14}")
print("-" * 52)
phases = [
    ("Random exploration",    episode_rewards[:100]),
    ("Early learning",        episode_rewards[100:500]),
    ("Convergence zone",      episode_rewards[500:1000]),
    ("Exploitation phase",    episode_rewards[2000:3000]),
    ("Final policy",          episode_rewards[-100:]),
]
for name, vals in phases:
    print(f"{name:<25} {len(vals):>10,} {np.mean(vals):>14.2f}")

In [ ]:
# ── Epsilon decay curve ──────────────────────────────────────────────────────
eps_curve = [max(0.05, 1.0 * (0.995**ep)) for ep in range(5000)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.5))
fig.patch.set_facecolor(C_BG)

# Epsilon decay
ax1.plot(eps_curve, color=C_WARN, lw=1.5)
ax1.axhline(0.05, color=C_RED, lw=1.5, ls="--", label="epsilon_min = 0.05")
ax1.set_xlabel("Episode"); ax1.set_ylabel("Epsilon")
ax1.set_title("Exploration Rate Decay", color="white")
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

# Raw reward
ax2.plot(episode_rewards, color=C_BLUE, lw=0.5, alpha=0.5, label="Episode reward")
ax2.axhline(0, color=C_RED, lw=1, ls="--", alpha=0.7)
ax2.set_xlabel("Episode"); ax2.set_ylabel("Total reward")
ax2.set_title("Episode Rewards (raw)", color="white")
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fig_02_training.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()
print("Left: as epsilon decays, the agent exploits learned knowledge more often.")
print("Right: reward rises from ~-820 to ~+85 over 5,000 episodes.")

## 6 · The Q-Table — What the Agent Remembers

After training, the Q-table is the agent's complete policy memory.
100 rows × 4 columns: one Q-value per state-action pair.

The Q-table is exported as `Data_AGV.csv` — a portable, inspectable record
of everything the agent learned.


In [ ]:
# ── Build and export Q-table ─────────────────────────────────────────────────
df_q = build_qtable_df(Q, env)
for col in ["Q_up","Q_down","Q_left","Q_right","best_Q"]:
    df_q[col] = df_q[col].round(2)

try:
    df_q.to_csv("Data_AGV.csv", index=False)
    print("Q-table saved: Data_AGV.csv")
except Exception as e:
    print(f"Save skipped ({e}) — continuing with in-memory Q-table.")

print(f"Shape: {df_q.shape}  |  {df_q.shape[0]} states x {df_q.shape[1]} columns")
print()
print("First 8 rows:")
display(df_q.head(8).round(2))

In [ ]:
# ── Load Q-table from CSV (portable workflow) ────────────────────────────────
try:
    df_q = pd.read_csv("Data_AGV.csv")
except FileNotFoundError:
    df_q = pd.read_csv(
        "https://raw.githubusercontent.com/LozanoLsa/"
        "QLearning_AGV/main/Data_AGV.csv"
    )

print(f"Q-table loaded: {df_q.shape}")
print()
print("Action distribution (best_action per cell):")
print(df_q["best_action"].value_counts().to_string())
print()
print("Best-Q statistics across all states:")
print(df_q["best_Q"].describe().round(2).to_string())

## 7 · Convergence — The Arc of Learning

Three analytical views of how the agent improved:

1. **Smoothed reward curve** — confirms convergence and identifies the learning phases
2. **Phase comparison** — quantifies the improvement from exploration to exploitation
3. **Convergence episode** — when does the agent first become reliably effective?


In [ ]:
# ── Smoothed convergence curve ───────────────────────────────────────────────
rewards_s = pd.Series(episode_rewards)
roll_50  = rewards_s.rolling(50).mean()
roll_200 = rewards_s.rolling(200).mean()

fig, ax = plt.subplots(figsize=(12, 4.5))
fig.patch.set_facecolor(C_BG)

ax.plot(episode_rewards, color=C_BLUE, lw=0.4, alpha=0.3, label="Episode reward")
ax.plot(roll_50,  color=C_WARN, lw=1.5, label="50-ep rolling mean")
ax.plot(roll_200, color=C_OK,   lw=2.0, label="200-ep rolling mean")
ax.axhline(0, color=C_RED, lw=1.0, ls="--", alpha=0.6, label="Break-even (reward=0)")

# Annotate convergence
converge_ep = int(roll_200[roll_200 > 0].index[0])
ax.axvline(converge_ep, color=C_WARN, lw=1.5, ls=":", alpha=0.8)
ax.text(converge_ep+30, -400, f"Ep {converge_ep}\nconvergence", color=C_WARN, fontsize=8)

ax.set_xlabel("Episode", fontsize=10)
ax.set_ylabel("Total Reward", fontsize=10)
ax.set_title("Q-Learning Convergence — From Blind Wandering to Optimal Navigation",
             color="white", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fig_03_convergence.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()
print(f"The agent crossed the break-even point at episode {converge_ep}.")
print(f"From there, it reliably reaches the goal faster than it accumulates penalties.")

In [ ]:
# ── Learning phase summary ───────────────────────────────────────────────────
phase_summary = pd.DataFrame({
    "Phase":       ["Episodes 1-100", "Episodes 100-500", "Episodes 500-1000",
                    "Episodes 1000-2000", "Episodes 2000-5000"],
    "Epsilon":     ["1.00 → 0.61", "0.61 → 0.08", "0.08 → 0.07", "0.07 → 0.05", "0.05 (floor)"],
    "Behavior":    ["Pure exploration", "Rapid learning", "Policy refinement",
                    "Exploitation dominant", "Near-optimal routing"],
    "Mean Reward": [
        f"{np.mean(episode_rewards[:100]):.1f}",
        f"{np.mean(episode_rewards[100:500]):.1f}",
        f"{np.mean(episode_rewards[500:1000]):.1f}",
        f"{np.mean(episode_rewards[1000:2000]):.1f}",
        f"{np.mean(episode_rewards[2000:]):.1f}",
    ]
}).set_index("Phase")
display(phase_summary)

print()
print(f"Total improvement: {np.mean(episode_rewards[:100]):.1f} -> {np.mean(episode_rewards[-100:]):.1f}")
print(f"(+{np.mean(episode_rewards[-100:]) - np.mean(episode_rewards[:100]):.1f} reward points over 5,000 episodes)")

## 8 · Optimal Policy — What the Agent Decided

The **policy** is the decision rule extracted from the Q-table:
for each cell, take the action with the highest Q-value.

Two views of the learned policy:
1. **Arrow grid**: the optimal direction at each cell
2. **Q* heatmap**: the maximum expected future reward at each cell (state value function)

High Q* values mark cells close to the goal. Low (or zero) Q* values mark obstacles
and cells the agent learned to avoid.


In [ ]:
# ── Optimal policy arrow grid ────────────────────────────────────────────────
action_symbols = {"up":"↑","down":"↓","left":"←","right":"→"}

fig, ax = plt.subplots(figsize=(7, 7))
fig.patch.set_facecolor(C_BG); ax.set_facecolor(C_CARD)
ax.set_xlim(0, env.cols); ax.set_ylim(0, env.rows)

for i in range(env.rows + 1): ax.axhline(i, color="#30363d", lw=0.4)
for j in range(env.cols + 1): ax.axvline(j, color="#30363d", lw=0.4)

for _, row_data in df_q.iterrows():
    r, c = int(row_data["row"]), int(row_data["col"])
    if (r, c) in env.obstacles:
        ax.fill_between([c,c+1], r, r+1, color=C_RED, alpha=0.7, zorder=2)
    elif (r, c) == env.goal:
        ax.fill_between([c,c+1], r, r+1, color=C_OK, alpha=0.7, zorder=2)
        ax.text(c+0.5, r+0.5, "G", ha="center", va="center",
                fontsize=11, color="white", fontweight="bold", zorder=4)
    else:
        sym = action_symbols.get(row_data["best_action"], "?")
        color = C_WARN if (r,c) in env.congestion_zones else C_BLUE
        ax.text(c+0.5, r+0.5, sym, ha="center", va="center",
                fontsize=12, color=color, fontweight="bold", zorder=3)

ax.invert_yaxis()
ax.set_xticks([j+0.5 for j in range(env.cols)])
ax.set_xticklabels(range(env.cols), fontsize=8)
ax.set_yticks([i+0.5 for i in range(env.rows)])
ax.set_yticklabels(range(env.rows), fontsize=8)
ax.set_title("Optimal Policy — Learned Action per Cell (AGV)", color="white", fontsize=12)

legend = [mpatches.Patch(color=C_RED,alpha=0.7,label="Obstacle"),
          mpatches.Patch(color=C_WARN,alpha=0.7,label="Congestion zone"),
          mpatches.Patch(color=C_OK,alpha=0.7,label="Goal")]
ax.legend(handles=legend,fontsize=8,loc="upper left")

plt.tight_layout()
plt.savefig("fig_04_policy.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()
print("Each arrow shows the agent's best decision from that cell.")
print("Arrows collectively form a flow field pointing toward the goal, routing around obstacles.")

In [ ]:
# ── Q* value heatmap ─────────────────────────────────────────────────────────
grid_bestQ = df_q.pivot(index="row", columns="col", values="best_Q")

fig, ax = plt.subplots(figsize=(7, 6))
fig.patch.set_facecolor(C_BG)

im = ax.imshow(grid_bestQ.values, cmap="viridis", origin="upper", aspect="equal")
plt.colorbar(im, ax=ax, label="Q* — Max expected future reward", shrink=0.85)

# Overlay obstacle markers
for (r,c) in env.obstacles:
    ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color=C_RED, alpha=0.7))

# Mark goal
gr, gc = env.goal
ax.text(gc, gr, "G", ha="center", va="center",
        fontsize=13, color="white", fontweight="bold")

ax.set_title("Q* Value Heatmap — Expected Future Reward per Cell", color="white", fontsize=12)
ax.set_xlabel("Column"); ax.set_ylabel("Row")
ax.set_xticks(range(10)); ax.set_yticks(range(10))

plt.tight_layout()
plt.savefig("fig_05_heatmap.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()
print("Bright cells: high future reward (close to goal, clear path).")
print("Dark cells: low Q* (obstacles, dead-ends, or far from goal).")
print(f"Q* range: {df_q['best_Q'].min():.2f} to {df_q['best_Q'].max():.2f}")

## 9 · Route Simulation — The Agent in Action

The trained policy is tested from three different start positions.
Each simulation follows the greedy policy: at every cell, take the action with highest Q-value.


In [ ]:
# ── Route simulator ──────────────────────────────────────────────────────────
def simulate_route(env, Q, start_rc, max_steps=100):
    """
    Follow greedy policy from start_rc = (row, col) to goal.
    Returns: (path, total_reward, reached_goal, n_steps)
    """
    r0, c0 = start_rc
    state = env.rc_to_state(r0, c0)
    path, total_r = [], 0.0

    for _ in range(max_steps):
        r, c = env.state_to_rc(state)
        path.append((r, c))
        if (r, c) == env.goal:
            break
        action = int(np.argmax(Q[state, :]))
        next_s, rew, done = env.step(state, action)
        total_r += rew; state = next_s
        if done:
            path.append(env.state_to_rc(state)); break

    reached = (env.state_to_rc(state) == env.goal)
    return path, total_r, reached, len(path) - 1

# ── Three scenarios ───────────────────────────────────────────────────────────
scenarios = [
    ((0, 0), "Scenario A — Top-left corner (maximum distance)"),
    ((0, 9), "Scenario B — Top-right corner (must bypass wall)"),
    ((5, 0), "Scenario C — Mid-left (congestion zone en route)"),
]

results = []
for (start_rc, name) in scenarios:
    path, reward, reached, steps = simulate_route(env, Q, start_rc)
    results.append({"Scenario": name, "Start": start_rc, "Steps": steps,
                    "Total Reward": round(reward, 1), "Reached Goal": reached})
    print(f"{name}")
    print(f"  Path: {path}")
    print(f"  Steps: {steps}  |  Reward: {reward:.1f}  |  Goal: {reached}")
    print()

summary_df = pd.DataFrame(results).set_index("Scenario")
display(summary_df)

In [ ]:
# ── Visualize all three routes on the factory floor ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
fig.patch.set_facecolor(C_BG)
fig.suptitle("AGV Route Simulations — Greedy Policy from 3 Start Positions",
             color="white", fontsize=13, y=1.02)

route_colors = [C_BLUE, C_WARN, C_OK]

for ax, (start_rc, name), color in zip(axes, scenarios, route_colors):
    path, reward, reached, steps = simulate_route(env, Q, start_rc)

    # Background grid
    for i in range(env.rows + 1): ax.axhline(i, color="#30363d", lw=0.4)
    for j in range(env.cols + 1): ax.axvline(j, color="#30363d", lw=0.4)
    ax.set_facecolor(C_CARD)
    ax.set_xlim(0, env.cols); ax.set_ylim(0, env.rows)

    # Obstacles & zones
    for (r,c) in env.obstacles:
        ax.fill_between([c,c+1], r, r+1, color=C_RED, alpha=0.6, zorder=2)
    for (r,c) in env.congestion_zones:
        ax.fill_between([c,c+1], r, r+1, color=C_WARN, alpha=0.3, zorder=2)

    # Goal
    gr, gc = env.goal
    ax.fill_between([gc,gc+1], gr, gr+1, color=C_OK, alpha=0.7, zorder=2)
    ax.text(gc+0.5, gr+0.5, "G", ha="center", va="center",
            fontsize=10, color="white", fontweight="bold", zorder=4)

    # Route
    xs = [c+0.5 for (r,c) in path]
    ys = [r+0.5 for (r,c) in path]
    ax.plot(xs, ys, color=color, lw=2.5, zorder=5, marker="o",
            markersize=4, markerfacecolor=color)
    sr, sc = path[0]
    ax.text(sc+0.5, sr+0.5, "S", ha="center", va="center",
            fontsize=9, color="white", fontweight="bold", zorder=6)

    ax.invert_yaxis()
    ax.set_xticks([]); ax.set_yticks([])
    label = name.split("—")[1].strip()
    ax.set_title(f"{label}\n{steps} steps | Reward: {reward:.0f}",
                 color="white", fontsize=9)

plt.tight_layout()
plt.savefig("fig_06_routes.png", dpi=130, bbox_inches="tight", facecolor=C_BG)
plt.show()

## 10 · Sensitivity Analysis — How Hyperparameters Shape the Agent

Three hyperparameters define the agent's learning dynamics:

| Parameter | Value used | Effect |
|-----------|-----------|--------|
| $\alpha$ (learning rate) | 0.1 | How fast Q-values update per step |
| $\gamma$ (discount factor) | 0.95 | How much future rewards are valued |
| Episodes | 5,000 | Total training budget |

We vary each one while holding the others fixed to measure impact on convergence.


In [ ]:
# ── Alpha sensitivity ─────────────────────────────────────────────────────────
print("Alpha (learning rate) sensitivity — mean reward in last 200 episodes:")
print(f"{'Alpha':>8} {'Mean Reward (last 200)':>25} {'Assessment':>20}")
print("-" * 58)

for alpha_test in [0.01, 0.05, 0.1, 0.2, 0.5]:
    np.random.seed(42)
    Q_t, r_t = q_learning(env, episodes=2000, alpha=alpha_test,
                           gamma=0.95, max_steps=200)
    mean_r = np.mean(r_t[-200:])
    assess = "optimal" if alpha_test == 0.1 else ("too slow" if alpha_test < 0.05 else "unstable" if alpha_test > 0.2 else "good")
    print(f"{alpha_test:>8.2f} {mean_r:>25.2f} {assess:>20}")

print()
print("alpha=0.1 balances fast initial learning with stable convergence.")

In [ ]:
# ── Gamma sensitivity ─────────────────────────────────────────────────────────
print("Gamma (discount factor) sensitivity — mean reward in last 200 episodes:")
print(f"{'Gamma':>8} {'Mean Reward (last 200)':>25} {'Assessment':>20}")
print("-" * 58)

for gamma_test in [0.7, 0.8, 0.9, 0.95, 0.99]:
    np.random.seed(42)
    Q_t, r_t = q_learning(env, episodes=2000, alpha=0.1,
                           gamma=gamma_test, max_steps=200)
    mean_r = np.mean(r_t[-200:])
    assess = "optimal" if gamma_test == 0.95 else ("myopic" if gamma_test < 0.85 else "near-optimal")
    print(f"{gamma_test:>8.2f} {mean_r:>25.2f} {assess:>20}")

print()
print("gamma=0.95 is critical for long-horizon planning in a 10x10 grid.")
print("Low gamma makes the agent myopic — it doesn't plan far enough ahead.")

## 11 · Simulator — Deploy the Learned Policy

The trained agent evaluates any starting cell and returns the optimal route,
step count, and total expected reward. Three scenarios demonstrate the range of situations
the agent handles confidently after training.


In [ ]:
# ── Interactive route evaluator ──────────────────────────────────────────────
def evaluate_start(start_row, start_col, verbose=True):
    """
    Evaluate the optimal route from a given starting cell.
    Returns route summary and visualizes the path.
    """
    if not env.is_valid(start_row, start_col):
        print(f"({start_row},{start_col}) is an obstacle or out of bounds. Choose a valid cell.")
        return

    path, reward, reached, steps = simulate_route(env, Q, (start_row, start_col))

    if verbose:
        print(f"Start  : ({start_row}, {start_col})")
        print(f"Goal   : {env.goal}")
        print(f"Steps  : {steps}")
        print(f"Reward : {reward:.1f}")
        print(f"Status : {'Reached goal' if reached else 'Did not reach goal (increase max_steps)'}")
        print(f"Route  : {path}")
        print()

    # Visualize
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    fig.patch.set_facecolor(C_BG)
    plot_grid(env, path=path,
              title=f"Start ({start_row},{start_col}) -> Goal | {steps} steps | Reward {reward:.0f}",
              ax=ax)
    plt.tight_layout()
    plt.show()
    return {"start": (start_row,start_col), "steps": steps, "reward": reward, "reached": reached}

In [ ]:
# ── Scenario A: standard run from top-left ───────────────────────────────────
print("="*55)
print("SCENARIO A — Standard run from top-left corner (0,0)")
print("="*55)
_ = evaluate_start(0, 0)

# ── Scenario B: starts near congestion zone ───────────────────────────────────
print("="*55)
print("SCENARIO B — Start near congestion zone (4, 3)")
print("="*55)
_ = evaluate_start(4, 3)

# ── Scenario C: starts near wall ──────────────────────────────────────────────
print("="*55)
print("SCENARIO C — Start adjacent to vertical wall (3, 3)")
print("="*55)
_ = evaluate_start(3, 3)

In [ ]:
# ── Final Reflection ──────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║     FINAL REFLECTION — Q-Learning & The Close of a Chapter            ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  Twenty-two projects. From the first logistic regression to this        ║
║  tabular Q-learning agent, we have crossed the full arc of machine      ║
║  learning: supervised, unsupervised, and now reinforcement.             ║
║                                                                          ║
║  Each paradigm answered a different question.                           ║
║  Supervised: given examples, what pattern holds?                        ║
║  Unsupervised: given unlabeled data, what structure exists?             ║
║  Reinforcement: given a world and a goal, what action is best now?      ║
║                                                                          ║
║  The AGV in this notebook doesn't know the factory floor at episode 1.  ║
║  It collides with walls. It wanders into congestion zones.              ║
║  It occasionally finds the goal by accident.                            ║
║                                                                          ║
║  By episode 5,000, it navigates cleanly: 19 steps from corner          ║
║  to dock, reward 83.0, no unnecessary collisions.                       ║
║                                                                          ║
║  That transformation — from blind wandering to purposeful action —      ║
║  is the closing insight of this chapter:                                ║
║                                                                          ║
║  Intelligence, whether biological or algorithmic,                        ║
║  is ultimately learned through consequence.                             ║
║                                                                          ║
║  The reward function is the teacher.                                    ║
║  Time is the curriculum.                                                ║
║  The Q-table is what remains when learning is done.                    ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")
print("LozanoLsa | Operational Excellence · MBB · Machine Learning | GitHub: LozanoLsa")